In [5]:

print("Punjab GeoGuardian Project")
print("Python environment is ready!")


Punjab GeoGuardian Project
Python environment is ready!


In [6]:

import sys

print("Python version:", sys.version)


Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]


In [7]:

%pip install -q earthengine-api geemap folium rasterio geopandas scikit-learn pandas numpy matplotlib seaborn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 41.3 MB/s eta 0:00:00


In [9]:
import ee
import geemap
import folium
import rasterio
import geopandas as gpd
import sklearn
import pandas as pd
import numpy as np
import matplotlib
import seaborn as sns

print("All required libraries imported successfully!")
print("Scikit-learn version:", sklearn.__version__)
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)


All required libraries imported successfully!
Scikit-learn version: 1.6.1
Pandas version: 2.2.3
NumPy version: 2.1.3


In [10]:

from pathlib import Path

folders = [
    "data",
    "outputs",
    "outputs/maps",
    "outputs/figures",
    "outputs/tables"
]

for folder in folders:
    Path(folder).mkdir(parents=True, exist_ok=True)

print("Project folders created successfully!")


Project folders created successfully!


In [11]:

import ee

print("Earth Engine library imported!")



Earth Engine library imported!


In [13]:

ee.Authenticate()


In [19]:

import ee

PROJECT_ID = "punjab-geoguardian-511815"

ee.Initialize(project=PROJECT_ID)

print("Earth Engine initialized successfully!")


Earth Engine initialized successfully!


In [20]:

test = ee.String("Hello from Google Earth Engine!")

print(test.getInfo())


Hello from Google Earth Engine!


In [23]:
# Load the updated administrative boundaries dataset
gaul = ee.FeatureCollection(
    "FAO/GAUL/2025/level2"
)

# Filter boundaries for Pakistan
pakistan = gaul.filter(
    ee.Filter.eq("GAUL0_NAME", "Pakistan")
)

# Get the available province names
province_names = pakistan.aggregate_array(
    "GAUL1_NAME"
).distinct().getInfo()

print("Available provinces in Pakistan:")
print(province_names)

Available provinces in Pakistan:
['Balochistan', 'Islamabad', 'Khyber Pakhtunkhwa', 'Punjab', 'Sindh']


In [28]:
# Load the updated administrative boundaries dataset
gaul = ee.FeatureCollection(
    "FAO/GAUL/2025/level2"
)

# Filter boundaries for Pakistan
pakistan = gaul.filter(
    ee.Filter.eq("GAUL0_NAME", "Pakistan")
)

# Filter boundaries for Punjab
punjab = pakistan.filter(
    ee.Filter.eq("GAUL1_NAME", "Punjab")
)

# Check the number of Punjab features
print("Punjab boundary features:", punjab.size().getInfo())

# Get the district names available in Punjab
district_names = punjab.aggregate_array(
    "GAUL2_NAME"
).distinct().getInfo()

print("Available districts in Punjab:")
print(district_names)

# Select Faisalabad district
faisalabad = punjab.filter(
    ee.Filter.eq("GAUL2_NAME", "Faisalabad")
)

district_count = faisalabad.size().getInfo()

print("Matching Faisalabad features:", district_count)

# Stop if the district was not found
if district_count == 0:
    raise ValueError(
        "Faisalabad was not found. Check the printed district names."
    )

# Create the Area of Interest (AOI)
aoi = faisalabad.geometry()

# Calculate approximate area in square kilometres
aoi_area_km2 = (
    aoi.area(maxError=100)
       .divide(1_000_000)
       .getInfo()
)

print(f"AOI area: {aoi_area_km2:,.2f} km²")
print("AOI created successfully!")

Punjab boundary features: 36
Available districts in Punjab:
['Multan', 'Muzaffargarh', 'Nankana Sahib', 'Narowal', 'Okara', 'Pakpattan', 'Rahim Yar Khan', 'Rajanpur', 'Rawalpindi', 'Sahiwal', 'Sargodha', 'Sheikhupura', 'Sialkot', 'Toba Tek Singh', 'Vehari', 'Attock', 'Bahawalnagar', 'Bahawalpur', 'Bhakkar', 'Chakwal', 'Chiniot', 'Dera Ghazi Khan', 'Faisalabad', 'Gujranwala', 'Gujrat', 'Hafizabad', 'Jhang', 'Jhelum', 'Kasur', 'Khanewal', 'Khushab', 'Lahore', 'Leiah', 'Lodhran', 'Mandi Bahauddin', 'Mianwali']
Matching Faisalabad features: 1
AOI area: 5,953.79 km²
AOI created successfully!


In [32]:
# Create an interactive map with an alternative basemap
Map = geemap.Map(
    basemap="Esri.WorldImagery"
)

# Center the map on Faisalabad
Map.centerObject(faisalabad, 8)

# Style the district boundary
boundary_layer = faisalabad.style(
    color="red",
    fillColor="00000000",
    width=2
)

# Add the district boundary
Map.addLayer(
    boundary_layer,
    {},
    "Faisalabad District Boundary"
)

# Display the map
Map

Map(center=[31.238674144990544, 73.14786051904862], controls=(WidgetControl(options=['position', 'transparent_…

In [33]:
# Load the Sentinel-2 Surface Reflectance collection
s2 = ee.ImageCollection(
    "COPERNICUS/S2_SR_HARMONIZED"
)

# Filter imagery for February-April 2019
s2_2019_all = (
    s2.filterBounds(aoi)
      .filterDate("2019-02-01", "2019-05-01")
)

# Filter imagery for February-April 2025
s2_2025_all = (
    s2.filterBounds(aoi)
      .filterDate("2025-02-01", "2025-05-01")
)

# Count matching scenes
count_2019_all = s2_2019_all.size().getInfo()
count_2025_all = s2_2025_all.size().getInfo()

print("2019 matching scenes:", count_2019_all)
print("2025 matching scenes:", count_2025_all)

2019 matching scenes: 120
2025 matching scenes: 141


In [34]:
# Filter scenes with less than 60% reported cloud cover
s2_2019 = s2_2019_all.filter(
    ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 60)
)

s2_2025 = s2_2025_all.filter(
    ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 60)
)

# Count scenes after filtering
count_2019 = s2_2019.size().getInfo()
count_2025 = s2_2025.size().getInfo()

print("2019 scenes after cloud filtering:", count_2019)
print("2025 scenes after cloud filtering:", count_2025)

2019 scenes after cloud filtering: 87
2025 scenes after cloud filtering: 125


In [35]:
# Create median composites and clip them to the AOI
composite_2019 = s2_2019.median().clip(aoi)
composite_2025 = s2_2025.median().clip(aoi)

print("2019 composite created successfully.")
print("2025 composite created successfully.")

2019 composite created successfully.
2025 composite created successfully.


In [36]:
# RGB visualization settings
rgb_vis = {
    "bands": ["B4", "B3", "B2"],
    "min": 0,
    "max": 3000
}

# Create an interactive map
Map = geemap.Map(
    basemap="Esri.WorldImagery"
)

# Center the map on Faisalabad
Map.centerObject(faisalabad, 8)

# Add the 2019 composite
Map.addLayer(
    composite_2019,
    rgb_vis,
    "Sentinel-2 RGB 2019",
    True
)

# Add the 2025 composite
Map.addLayer(
    composite_2025,
    rgb_vis,
    "Sentinel-2 RGB 2025",
    False
)

# Add the district boundary
Map.addLayer(
    faisalabad.style(
        color="red",
        fillColor="00000000",
        width=2
    ),
    {},
    "Faisalabad District Boundary"
)

# Display the map
Map

Map(center=[31.238674144990544, 73.14786051904862], controls=(WidgetControl(options=['position', 'transparent_…

In [37]:

# Define a function to mask clouds and unwanted pixels
def mask_s2_clouds(image):
    scl = image.select("SCL")

    # Mask cloud shadows, clouds, cirrus, snow, and defective pixels
    mask = (
        scl.neq(1)
        .And(scl.neq(3))
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11))
    )

    # Keep the spectral bands needed for our analysis
    return (
        image.updateMask(mask)
        .select(["B2", "B3", "B4", "B8", "B11", "B12"])
        .copyProperties(image, ["system:time_start"])
    )


# Apply cloud masking to both years
s2_2019_clean = s2_2019.map(mask_s2_clouds)
s2_2025_clean = s2_2025.map(mask_s2_clouds)

# Create new cloud-masked median composites
composite_2019_clean = s2_2019_clean.median().clip(aoi)
composite_2025_clean = s2_2025_clean.median().clip(aoi)

print("Cloud masking completed successfully!")
print("2019 images:", s2_2019_clean.size().getInfo())
print("2025 images:", s2_2025_clean.size().getInfo())


Cloud masking completed successfully!
2019 images: 87
2025 images: 125


In [39]:

Map = geemap.Map(basemap="HYBRID")
Map.centerObject(aoi, 8)

Map.addLayer(
    composite_2019_clean,
    rgb_vis,
    "Cloud-Masked RGB 2019",
    True
)

Map.addLayer(
    composite_2025_clean,
    rgb_vis,
    "Cloud-Masked RGB 2025",
    False
)

Map.addLayer(
    faisalabad.style(
        color="red",
        fillColor="00000000",
        width=2
    ),
    {},
    "Faisalabad District Boundary"
)

Map.add_layer_control()
Map


Map(center=[31.238674144990576, 73.14786051904862], controls=(WidgetControl(options=['position', 'transparent_…

In [40]:

# Calculate NDVI for 2019 and 2025
ndvi_2019 = composite_2019_clean.normalizedDifference(
    ["B8", "B4"]
).rename("NDVI_2019")

ndvi_2025 = composite_2025_clean.normalizedDifference(
    ["B8", "B4"]
).rename("NDVI_2025")


# Calculate NDBI for 2019 and 2025
ndbi_2019 = composite_2019_clean.normalizedDifference(
    ["B11", "B8"]
).rename("NDBI_2019")

ndbi_2025 = composite_2025_clean.normalizedDifference(
    ["B11", "B8"]
).rename("NDBI_2025")


print("NDVI and NDBI calculated successfully!")


NDVI and NDBI calculated successfully!


In [42]:

Map = geemap.Map(basemap="ROADMAP")
Map.centerObject(aoi, 8)

Map.addLayer(
    composite_2019_clean,
    rgb_vis,
    "Satellite Image 2019",
    True
)

Map.addLayer(
    composite_2025_clean,
    rgb_vis,
    "Satellite Image 2025",
    False
)

Map.addLayer(
    faisalabad.style(
        color="red",
        fillColor="00000000",
        width=2
    ),
    {},
    "District Boundary"
)

Map.add_layer_control()
Map


Map(center=[31.238674144990576, 73.14786051904862], controls=(WidgetControl(options=['position', 'transparent_…

In [43]:

# Prepare satellite bands for classification

bands = ["B2", "B3", "B4", "B8", "B11", "B12"]

# Add vegetation and built-up indices
image_2019 = composite_2019_clean.addBands(ndvi_2019).addBands(ndbi_2019)
image_2025 = composite_2025_clean.addBands(ndvi_2025).addBands(ndbi_2025)

# Select the same predictor bands for both years
predictor_bands = bands + ["NDVI_2019", "NDBI_2019"]

print("2019 image prepared:", image_2019.bandNames().getInfo())
print("2025 image prepared:", image_2025.bandNames().getInfo())
print("Predictor bands prepared successfully!")


2019 image prepared: ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDVI_2019', 'NDBI_2019']
2025 image prepared: ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDVI_2025', 'NDBI_2025']
Predictor bands prepared successfully!


In [44]:

# Define land-cover class labels
CLASS_OTHER = 0
CLASS_VEGETATION = 1
CLASS_BUILT_UP = 2

# Define the predictor bands
predictor_bands = [
    "B2", "B3", "B4", "B8", "B11", "B12",
    "NDVI_2019", "NDBI_2019"
]

# Prepare the 2019 image for training
training_image = image_2019.select(predictor_bands)

print("Training image prepared!")
print("Predictor bands:", predictor_bands)
print("Class labels:")
print("0 = Other land")
print("1 = Vegetation / Agriculture")
print("2 = Built-up")


Training image prepared!
Predictor bands: ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDVI_2019', 'NDBI_2019']
Class labels:
0 = Other land
1 = Vegetation / Agriculture
2 = Built-up


In [49]:

# Load ESA WorldCover 2021 reference map
worldcover = ee.ImageCollection("ESA/WorldCover/v200").first().select("Map")

# Map original WorldCover classes to our 3 project classes
# Class 0 = Other land
# Class 1 = Vegetation / Agriculture
# Class 2 = Built-up

original_classes = [
    10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 100
]

project_classes = [
    1, 1, 1, 1, 2, 0, 0, 0, 1, 1, 1
]

labels = (
    worldcover.remap(original_classes, project_classes)
    .rename("landcover")
    .clip(aoi)
)

# Combine reference labels with 2019 predictor image
training_stack = training_image.addBands(labels)

# Collect up to 100 samples per class
samples = training_stack.stratifiedSample(
    numPoints=100,
    classBand="landcover",
    region=aoi,
    scale=10,
    seed=42,
    geometries=True,
    tileScale=4
)

print("Training samples collected:", samples.size().getInfo())
print(
    "Samples per class:",
    samples.aggregate_histogram("landcover").getInfo()
)


# Create a map to visually inspect training labels
Map = geemap.Map()
Map.centerObject(aoi, 8)

# Show the 2019 satellite image
Map.addLayer(
    composite_2019_clean,
    {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000},
    "Sentinel-2 RGB 2019",
    True
)

# Show the reference land-cover labels
Map.addLayer(
    labels,
    {"min": 0, "max": 2, "palette": ["blue", "green", "red"]},
    "WorldCover Reference Labels",
    False
)

# Show sample points for each class
Map.addLayer(
    samples.filter(ee.Filter.eq("landcover", 0)).style(
        color="0000FF", pointSize=5, width=1
    ),
    {},
    "Class 0 - Other Land"
)

Map.addLayer(
    samples.filter(ee.Filter.eq("landcover", 1)).style(
        color="00FF00", pointSize=5, width=1
    ),
    {},
    "Class 1 - Vegetation / Agriculture"
)

Map.addLayer(
    samples.filter(ee.Filter.eq("landcover", 2)).style(
        color="FF0000", pointSize=5, width=1
    ),
    {},
    "Class 2 - Built-up"
)

Map.addLayer(
    faisalabad.style(
        color="FFFFFF",
        fillColor="00000000",
        width=2
    ),
    {},
    "Faisalabad District Boundary"
)

Map.add_layer_control()
Map


Training samples collected: 300
Samples per class: {'0': 100, '1': 100, '2': 100}


Map(center=[31.238674144990544, 73.14786051904862], controls=(WidgetControl(options=['position', 'transparent_…

In [46]:

# Step 1: Split the samples into training and testing sets
samples = samples.randomColumn("random", seed=42)

training_samples = samples.filter(
    ee.Filter.lt("random", 0.7)
)

testing_samples = samples.filter(
    ee.Filter.gte("random", 0.7)
)

print("Training samples:", training_samples.size().getInfo())
print("Testing samples:", testing_samples.size().getInfo())


# Step 2: Create the Random Forest classifier
classifier = ee.Classifier.smileRandomForest(
    numberOfTrees=100,
    seed=42
).train(
    features=training_samples,
    classProperty="landcover",
    inputProperties=predictor_bands
)

print("Random Forest model trained successfully!")


# Step 3: Classify the 2019 satellite image
classified_2019 = image_2019.select(
    predictor_bands
).classify(classifier).rename("landcover")

print("2019 land-cover classification completed!")


Training samples: 217
Testing samples: 83
Random Forest model trained successfully!
2019 land-cover classification completed!


In [47]:

# Predict classes for the testing samples
test_predictions = testing_samples.classify(classifier)

# Create the confusion matrix
confusion_matrix = test_predictions.errorMatrix(
    "landcover",
    "classification"
)

# Calculate evaluation metrics
overall_accuracy = confusion_matrix.accuracy()
kappa_score = confusion_matrix.kappa()
producer_accuracy = confusion_matrix.producersAccuracy()
consumer_accuracy = confusion_matrix.consumersAccuracy()

# Display the results
print("Confusion Matrix:")
print(confusion_matrix.getInfo())

print("Overall Accuracy:", overall_accuracy.getInfo())
print("Kappa Score:", kappa_score.getInfo())
print("Producer Accuracy:", producer_accuracy.getInfo())
print("Consumer Accuracy:", consumer_accuracy.getInfo())


Confusion Matrix:
[[17, 5, 8], [1, 19, 3], [12, 2, 16]]
Overall Accuracy: 0.6265060240963856
Kappa Score: 0.43833224186858766
Producer Accuracy: [[0.5666666666666667], [0.8260869565217391], [0.5333333333333333]]
Consumer Accuracy: [[0.5666666666666667, 0.7307692307692307, 0.5925925925925926]]


In [50]:

# Prepare the 2025 image using the same predictor names as training
image_2025_for_classification = (
    composite_2025_clean
    .addBands(ndvi_2025.rename("NDVI_2019"))
    .addBands(ndbi_2025.rename("NDBI_2019"))
)

# Classify the 2025 image using the trained Random Forest model
classified_2025 = (
    image_2025_for_classification
    .select(predictor_bands)
    .classify(classifier)
    .rename("landcover")
)

print("2025 land-cover classification completed!")


2025 land-cover classification completed!


In [51]:

class_vis = {
    "min": 0,
    "max": 2,
    "palette": ["blue", "green", "red"]
}

Map = geemap.Map()
Map.centerObject(aoi, 8)

Map.addLayer(
    classified_2019,
    class_vis,
    "Land Cover 2019",
    True
)

Map.addLayer(
    classified_2025,
    class_vis,
    "Land Cover 2025",
    False
)

Map.addLayer(
    faisalabad.style(
        color="white",
        fillColor="00000000",
        width=2
    ),
    {},
    "Faisalabad Boundary"
)

Map.add_layer_control()
Map


Map(center=[31.238674144990544, 73.14786051904862], controls=(WidgetControl(options=['position', 'transparent_…

In [52]:

# Calculate the area of each land-cover class in square kilometers
def calculate_class_areas(classified_image, year):
    area_image = (
        ee.Image.pixelArea()
        .divide(1_000_000)
        .rename("area_km2")
        .addBands(classified_image.select("landcover").rename("class"))
    )

    result = area_image.reduceRegion(
        reducer=ee.Reducer.sum().group(
            groupField=1,
            groupName="class"
        ),
        geometry=aoi,
        scale=10,
        maxPixels=1e13,
        tileScale=4
    )

    groups = result.get("groups").getInfo()

    print(f"\nLand-cover area statistics for {year}:")
    for item in sorted(groups, key=lambda x: x["class"]):
        class_id = int(item["class"])
        area_km2 = item["sum"]

        class_name = {
            0: "Other land",
            1: "Vegetation / Agriculture",
            2: "Built-up"
        }[class_id]

        print(f"{class_name}: {area_km2:,.2f} km²")


calculate_class_areas(classified_2019, 2019)
calculate_class_areas(classified_2025, 2025)



Land-cover area statistics for 2019:
Other land: 500.74 km²
Vegetation / Agriculture: 4,643.94 km²
Built-up: 803.89 km²

Land-cover area statistics for 2025:
Other land: 475.11 km²
Vegetation / Agriculture: 4,634.47 km²
Built-up: 838.99 km²


In [53]:

# Identify pixels classified as vegetation in 2019
# and built-up in 2025
vegetation_to_builtup = (
    classified_2019.select("landcover").eq(1)
    .And(classified_2025.select("landcover").eq(2))
)

# Calculate the estimated area of these pixels
change_area = (
    ee.Image.pixelArea()
    .divide(1_000_000)
    .updateMask(vegetation_to_builtup)
    .rename("area_km2")
    .reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=aoi,
        scale=10,
        maxPixels=1e13,
        tileScale=4
    )
)

estimated_change_km2 = change_area.get("area_km2").getInfo()

print(
    "Estimated vegetation-to-built-up change:",
    round(estimated_change_km2 or 0, 2),
    "km²"
)


Estimated vegetation-to-built-up change: 344.84 km²


In [54]:

# Identify pixels that changed from vegetation/agriculture to built-up.
vegetation_2019 = classified_2019.eq(1)
built_up_2025 = classified_2025.eq(2)

vegetation_to_built_up = (
    vegetation_2019
    .And(built_up_2025)
    .selfMask()
    .rename("change")
)

# Display the estimated change on the map.
Map.addLayer(
    vegetation_to_built_up,
    {"palette": ["red"]},
    "Estimated Vegetation-to-Built-up Change"
)

Map


Map(bottom=53789.0, center=[31.38881292545334, 73.52253398393749], controls=(WidgetControl(options=['position'…

In [55]:

# Generate 60 candidate validation points.
validation_points = ee.FeatureCollection.randomPoints(
    region=aoi,
    points=60,
    seed=2025,
    maxError=100
)

print("Candidate validation points:", validation_points.size().getInfo())

# Display the candidate points on the map.
Map.addLayer(
    validation_points,
    {"color": "yellow"},
    "Candidate Validation Points"
)

Map


Candidate validation points: 60


Map(bottom=428261.0, center=[31.3744509892913, 73.14990868384069], controls=(WidgetControl(options=['position'…

In [61]:

# Generate 60 candidate validation points.
validation_points = ee.FeatureCollection.randomPoints(
    region=aoi,
    points=60,
    seed=2025,
    maxError=100
)

print("Candidate validation points:", validation_points.size().getInfo())

# Display the candidate points on the map.
Map.addLayer(
    validation_points,
    {"color": "yellow"},
    "Candidate Validation Points"
)

Map


Candidate validation points: 60


Map(bottom=13465.0, center=[33.31216783738621, 74.91027832031251], controls=(WidgetControl(options=['position'…

In [57]:

import pandas as pd

validation_table = pd.DataFrame({
    "point_id": range(1, 61),
    "reference_class": [None] * 60,
    "notes": [""] * 60
})

validation_table.to_csv("validation_points.csv", index=False)

print(validation_table.head())
print("Total candidate points:", len(validation_table))


   point_id reference_class notes
0         1            None      
1         2            None      
2         3            None      
3         4            None      
4         5            None      
Total candidate points: 60


In [62]:
import pandas as pd
from google.colab import files

# 60 validation points par 2025 classification prediction extract karna
validation_with_prediction = classified_2025.sampleRegions(
    collection=validation_points,
    scale=10,
    geometries=True
)

# Har point ke coordinates add karna
def add_coordinates(feature):
    coordinates = feature.geometry().coordinates()

    return feature.set({
        "longitude": coordinates.get(0),
        "latitude": coordinates.get(1)
    })

validation_with_prediction = validation_with_prediction.map(
    add_coordinates
)

# Earth Engine results ko Python list mein convert karna
features = validation_with_prediction.getInfo()["features"]

rows = [
    feature["properties"]
    for feature in features
]

# DataFrame banana
validation_df = pd.DataFrame(rows)

# Prediction column ka naam simple rakhna
validation_df = validation_df.rename(
    columns={
        "landcover": "predicted_class"
    }
)

# Manual checking ke liye blank columns
validation_df["reference_class"] = ""
validation_df["correct"] = ""
validation_df["notes"] = ""

# CSV save karna
validation_df.to_csv(
    "validation_points_with_predictions.csv",
    index=False
)

print("Validation CSV successfully created!")
print("Total validation points:", len(validation_df))
display(validation_df.head())

# CSV download karna
files.download("validation_points_with_predictions.csv")


Validation CSV successfully created!
Total validation points: 60


,predicted_class,latitude,longitude,reference_class,correct,notes
0,0,31.374784,73.214897,,,
1,2,31.485906,73.142582,,,
2,1,31.557681,73.254961,,,
3,0,31.613467,73.353507,,,
4,1,31.352775,73.575750,,,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [64]:
from google.colab import files

files.download("validation_points_with_predictions.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [66]:
import pandas as pd
import folium
from IPython.display import display
from google.colab import files

# CSV upload karein
uploaded = files.upload()

csv_name = list(uploaded.keys())[0]
validation_df = pd.read_csv(csv_name)

# Point ID add karein
if "point_id" not in validation_df.columns:
    validation_df.insert(
        0,
        "point_id",
        range(1, len(validation_df) + 1)
    )

# Class names
class_names = {
    0: "Other",
    1: "Vegetation / Agriculture",
    2: "Built-up / Urban"
}

# Correct point colors
colors = {
    0: "blue",
    1: "green",
    2: "red"
}

# Map create karein
validation_map = folium.Map(
    location=[
        validation_df["latitude"].mean(),
        validation_df["longitude"].mean()
    ],
    zoom_start=9,
    tiles=None
)

# Satellite layer
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/"
          "World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Imagery",
    name="Satellite Imagery",
    overlay=False,
    control=True
 ).add_to(validation_map)

# OpenStreetMap layer
folium.TileLayer(
    tiles="OpenStreetMap",
    name="OpenStreetMap",
    overlay=False,
    control=True
).add_to(validation_map)

# Points add karein
for _, row in validation_df.iterrows():

    predicted_class = int(row["predicted_class"])
    point_color = colors[predicted_class]

    popup_text = f"""
    <b>Point ID:</b> {row['point_id']}

    <b>Latitude:</b> {row['latitude']:.6f}

    <b>Longitude:</b> {row['longitude']:.6f}

    <b>Predicted Class:</b> {predicted_class}

    <b>Class Name:</b> {class_names[predicted_class]}
    """

    folium.CircleMarker(
        location=[
            row["latitude"],
            row["longitude"]
        ],
        radius=6,
        color=point_color,
        fill=True,
        fill_color=point_color,
        fill_opacity=0.9,
        popup=folium.Popup(
            popup_text,
            max_width=300
        )
    ).add_to(validation_map)

# Legend
legend_html = """
<div style="
position: fixed;
bottom: 30px;
left: 30px;
width: 220px;
background-color: white;
border: 2px solid grey;
z-index: 9999;
font-size: 14px;
padding: 10px;
">
<b>Validation Point Classes</b>

<span style="color:blue;">●</span>
0 = Other

<span style="color:green;">●</span>
1 = Vegetation / Agriculture

<span style="color:red;">●</span>
2 = Built-up / Urban
</div>
"""

validation_map.get_root().html.add_child(
    folium.Element(legend_html)
)

folium.LayerControl().add_to(validation_map)

display(validation_map)


Saving validation_points_with_predictions.csv to validation_points_with_predictions (2).csv


In [70]:
import pandas as pd
import folium
from IPython.display import display

# CSV read karein
validation_df = pd.read_csv(
    "validation_points_with_predictions.csv"
)

# Point ID create karein
if "point_id" not in validation_df.columns:
    validation_df.insert(
        0,
        "point_id",
        range(1, len(validation_df) + 1)
    )

# Map center
center_lat = validation_df["latitude"].mean()
center_lon = validation_df["longitude"].mean()

# Blank map
satellite_map = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=9,
    tiles=None
)

# Google Satellite imagery
folium.TileLayer(
    tiles="https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",
    attr="Google Satellite",
    name="Satellite",
    overlay=False,
    control=True
 ).add_to(satellite_map)

# OpenStreetMap alternative
folium.TileLayer(
    tiles="OpenStreetMap",
    name="Street Map",
    overlay=False,
    control=True
).add_to(satellite_map)

# Correct colors:
# 0 = Other = Blue
# 1 = Vegetation = Green
# 2 = Built-up = Red
point_colors = {
    0: "blue",
    1: "green",
    2: "red"
}

# All validation points add karein
for _, row in validation_df.iterrows():

    predicted = int(row["predicted_class"])
    point_id = int(row["point_id"])

    popup_text = f"""
    <b>Point ID:</b> {point_id}

    <b>Predicted class:</b> {predicted}

    <b>Latitude:</b> {row['latitude']:.6f}

    <b>Longitude:</b> {row['longitude']:.6f}
    """

    folium.CircleMarker(
        location=[
            row["latitude"],
            row["longitude"]
        ],
        radius=7,
        color=point_colors[predicted],
        fill=True,
        fill_color=point_colors[predicted],
        fill_opacity=1,
        popup=folium.Popup(
            popup_text,
            max_width=300
        ),
        tooltip=f"Point {point_id}"
    ).add_to(satellite_map)

folium.LayerControl().add_to(satellite_map)

display(satellite_map)


In [84]:
import pandas as pd
import folium
from folium.features import DivIcon
from IPython.display import display

validation_df = pd.read_csv("validation_points_with_predictions.csv")

if "point_id" not in validation_df.columns:
    validation_df.insert(0, "point_id", range(1, len(validation_df) + 1))

center_lat = validation_df["latitude"].mean()
center_lon = validation_df["longitude"].mean()

classification_map = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=9,
    width="100%",
    height=700,
    tiles=None
)

# Sentinel-2 RGB background
rgb_visualization = {
    "bands": ["B4", "B3", "B2"],
    "min": 0,
    "max": 3000
}

rgb_map_id = composite_2025_clean.getMapId(rgb_visualization)

folium.TileLayer(
    tiles=rgb_map_id["tile_fetcher"].url_format,
    attr="Google Earth Engine Sentinel-2",
    name="Sentinel-2 RGB 2025",
    overlay=False,
    control=True
).add_to(classification_map)

# Land classification:
# Blue = Other, Green = Vegetation, Red = Built-up
classification_visualization = {
    "min": 0,
    "max": 2,
    "palette": ["0000FF", "00AA00", "FF0000"],
    "opacity": 0.70
}

classification_map_id = classified_2025.getMapId(
    classification_visualization
)

folium.TileLayer(
    tiles=classification_map_id["tile_fetcher"].url_format,
    attr="Google Earth Engine Classification",
    name="Land Classification 2025",
    overlay=True,
    control=True
).add_to(classification_map)

# Add yellow dots and numbered labels
for _, row in validation_df.iterrows():

    point_id = int(row["point_id"])
    latitude = float(row["latitude"])
    longitude = float(row["longitude"])

    # Yellow validation dot
    folium.CircleMarker(
        location=[latitude, longitude],
        radius=7,
        color="yellow",
        weight=3,
        fill=True,
        fill_color="yellow",
        fill_opacity=1.0,
        tooltip="Point " + str(point_id),
        popup="Validation Point " + str(point_id)
    ).add_to(classification_map)

    # Number above the yellow dot
    number_html = '<div style="font-size:14px;font-weight:bold;color:white;background-color:black;border:2px solid yellow;border-radius:50%;width:26px;height:26px;line-height:26px;text-align:center;">' + str(point_id) + '</div>'

    folium.Marker(
        location=[latitude, longitude],
        icon=DivIcon(
            html=number_html,
            icon_size=(26, 26),
            icon_anchor=(13, 34)
        )
    ).add_to(classification_map)

folium.LayerControl().add_to(classification_map)

display(classification_map)


In [85]:
# Display all validation points in a simple table

display(
    validation_df[
        [
            "point_id",
            "latitude",
            "longitude",
            "predicted_class"
        ]
    ].head(60)
)


,point_id,latitude,longitude,predicted_class
0,1,31.374784,73.214897,0
1,2,31.485906,73.142582,2
2,3,31.557681,73.254961,1
3,4,31.613467,73.353507,0
4,5,31.352775,73.575750,1
5,6,30.868314,72.844611,1
6,7,31.430480,73.276970,2
7,8,31.297978,73.475318,1
8,9,31.373167,72.996067,2
9,10,31.425719,73.443608,1


In [87]:
import pandas as pd
from IPython.display import display

# Load the validation CSV
validation_df = pd.read_csv(
    "validation_points_with_predictions.csv"
)

# Add point IDs if they are missing
if "point_id" not in validation_df.columns:
    validation_df.insert(
        0,
        "point_id",
        range(1, len(validation_df) + 1)
    )

# Your manually checked reference classes
reference_values = [
    0, 2, 1, 0, 1, 1, 2, 1, 2, 1,
    1, 1, 1, 1, 2, 1, 0, 1, 0, 2,
    1, 1, 1, 2, 1, 1, 1, 1, 2, 2,
    1, 2, 1, 1, 1, 1, 1, 1, 1, 0,
    1, 1, 1, 2, 1, 1, 1, 1, 1, 2,
    2, 2, 0, 0, 1, 1, 1, 1, 1, 1
]

print("Reference values:", len(reference_values))
print("Validation rows:", len(validation_df))

# Check that the number of values matches the number of rows
if len(reference_values) != len(validation_df):
    raise ValueError(
        "The number of reference values must equal the number of validation rows."
    )

# Check that only valid class codes are used
if not all(value in [0, 1, 2] for value in reference_values):
    raise ValueError(
        "Only 0, 1, and 2 are allowed."
    )

# Add manual reference classes
validation_df["reference_class"] = reference_values

# Compare model predictions with manual reference classes
validation_df["correct"] = (
    validation_df["predicted_class"]
    == validation_df["reference_class"]
)

# Add readable class names
class_names = {
    0: "Other",
    1: "Vegetation / Agriculture",
    2: "Built-up / Urban"
}

validation_df["predicted_class_name"] = (
    validation_df["predicted_class"].map(class_names)
)

validation_df["reference_class_name"] = (
    validation_df["reference_class"].map(class_names)
)

# Calculate validation results
total_points = len(validation_df)
correct_points = int(validation_df["correct"].sum())
incorrect_points = total_points - correct_points
accuracy = correct_points / total_points

print("Validation completed successfully.")
print("Total points checked:", total_points)
print("Correct predictions:", correct_points)
print("Incorrect predictions:", incorrect_points)
print("Validation accuracy:", round(accuracy * 100, 2), "%")

# Display confusion matrix
print("\nConfusion Matrix:")

display(
    pd.crosstab(
        validation_df["reference_class_name"],
        validation_df["predicted_class_name"],
        rownames=["Actual Class"],
        colnames=["Predicted Class"]
    )
)

# Save completed CSV
validation_df.to_csv(
    "validation_points_completed.csv",
    index=False
)

print("\nSaved file: validation_points_completed.csv")


Reference values: 60
Validation rows: 60
Validation completed successfully.
Total points checked: 60
Correct predictions: 58
Incorrect predictions: 2
Validation accuracy: 96.67 %

Confusion Matrix:


Predicted Class,Built-up / Urban,Other,Vegetation / Agriculture
Actual Class,,,
Built-up / Urban,11,1,1
Other,0,7,0
Vegetation / Agriculture,0,0,40



Saved file: validation_points_completed.csv


In [88]:
from google.colab import files

files.download("validation_points_completed.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [89]:
from google.colab import files

uploaded = files.upload()


Saving validation_points_completed.csv to validation_points_completed (1).csv


In [90]:
import pandas as pd

validation_df = pd.read_csv(
    "validation_points_completed.csv"
)

print("Rows:", len(validation_df))
print(validation_df.head())


Rows: 60
   point_id  predicted_class   latitude  longitude  reference_class  correct  \
0         1                0  31.374784  73.214897                0     True   
1         2                2  31.485906  73.142582                2     True   
2         3                1  31.557681  73.254961                1     True   
3         4                0  31.613467  73.353507                0     True   
4         5                1  31.352775  73.575750                1     True   

   notes      predicted_class_name      reference_class_name  
0    NaN                     Other                     Other  
1    NaN          Built-up / Urban          Built-up / Urban  
2    NaN  Vegetation / Agriculture  Vegetation / Agriculture  
3    NaN                     Other                     Other  
4    NaN  Vegetation / Agriculture  Vegetation / Agriculture  


In [91]:
from google.colab import files
import pandas as pd

# Upload the completed validation CSV
uploaded = files.upload()

# Read the uploaded file
file_name = list(uploaded.keys())[0]
validation_df = pd.read_csv(file_name)

print("File loaded successfully.")
print("Total validation points:", len(validation_df))


Saving validation_points_completed.csv to validation_points_completed (2).csv
File loaded successfully.
Total validation points: 60


In [92]:
import pandas as pd

# Select the incorrect validation points first
incorrect_points = validation_df[
    validation_df["correct"] == False
].copy()

# Add more points so that we have 15 cases for error analysis
remaining_points = validation_df[
    validation_df["correct"] == True
].head(15 - len(incorrect_points))

error_analysis = pd.concat(
    [incorrect_points, remaining_points],
    ignore_index=True
)

# Keep exactly 15 cases
error_analysis = error_analysis.head(15).copy()

# Add case IDs
error_analysis.insert(
    0,
    "case_id",
    range(1, len(error_analysis) + 1)
)

# Add analysis columns
error_analysis["error_type"] = ""
error_analysis["possible_reason"] = ""
error_analysis["improvement"] = ""

# Save the error-analysis template
error_analysis.to_csv(
    "error_analysis_15_cases.csv",
    index=False
)

print("Error-analysis file created.")
print("Number of cases:", len(error_analysis))

display(
    error_analysis[
        [
            "case_id",
            "point_id",
            "predicted_class_name",
            "reference_class_name",
            "correct",
            "error_type",
            "possible_reason",
            "improvement"
        ]
    ]
)


Error-analysis file created.
Number of cases: 15


,case_id,point_id,predicted_class_name,reference_class_name,correct,error_type,possible_reason,improvement
0,1,29,Vegetation / Agriculture,Built-up / Urban,False,,,
1,2,44,Other,Built-up / Urban,False,,,
2,3,1,Other,Other,True,,,
3,4,2,Built-up / Urban,Built-up / Urban,True,,,
4,5,3,Vegetation / Agriculture,Vegetation / Agriculture,True,,,
5,6,4,Other,Other,True,,,
6,7,5,Vegetation / Agriculture,Vegetation / Agriculture,True,,,
7,8,6,Vegetation / Agriculture,Vegetation / Agriculture,True,,,
8,9,7,Built-up / Urban,Built-up / Urban,True,,,
9,10,8,Vegetation / Agriculture,Vegetation / Agriculture,True,,,


In [93]:
import pandas as pd

# Load the error-analysis file
error_analysis = pd.read_csv(
    "error_analysis_15_cases.csv"
)

# Class names
class_names = {
    0: "Other",
    1: "Vegetation / Agriculture",
    2: "Built-up / Urban"
}

# Fill explanations row by row
for index, row in error_analysis.iterrows():

    predicted = row["predicted_class_name"]
    reference = row["reference_class_name"]
    is_correct = bool(row["correct"])

    if not is_correct:
        error_analysis.loc[index, "error_type"] = (
            "Missed built-up area"
        )

        error_analysis.loc[index, "possible_reason"] = (
            "Vegetation, tree cover, mixed pixels, or low "
            "10-metre resolution may have caused the urban "
            "area to be classified as vegetation."
        )

        error_analysis.loc[index, "improvement"] = (
            "Add more manually verified built-up training samples, "
            "use higher-resolution imagery, and include texture "
            "features or additional land-cover classes."
        )

    else:
        error_analysis.loc[index, "error_type"] = (
            "Correct classification check"
        )

        error_analysis.loc[index, "possible_reason"] = (
            "The predicted class agrees with the manually checked "
            "reference class at this validation point."
        )

        error_analysis.loc[index, "improvement"] = (
            "Continue validation with more points and independent "
            "high-resolution imagery."
        )

# Save the completed error-analysis file
error_analysis.to_csv(
    "error_analysis_15_cases_completed.csv",
    index=False
)

print("Completed error-analysis file saved.")
print("Total analyzed cases:", len(error_analysis))
print(
    "Actual incorrect cases:",
    int((error_analysis["correct"] == False).sum())
)

display(
    error_analysis[
        [
            "case_id",
            "point_id",
            "predicted_class_name",
            "reference_class_name",
            "correct",
            "error_type",
            "possible_reason",
            "improvement"
        ]
    ]
)


Completed error-analysis file saved.
Total analyzed cases: 15
Actual incorrect cases: 2


,case_id,point_id,predicted_class_name,reference_class_name,correct,error_type,possible_reason,improvement
0,1,29,Vegetation / Agriculture,Built-up / Urban,False,Missed built-up area,"Vegetation, tree cover, mixed pixels, or low 1...",Add more manually verified built-up training s...
1,2,44,Other,Built-up / Urban,False,Missed built-up area,"Vegetation, tree cover, mixed pixels, or low 1...",Add more manually verified built-up training s...
2,3,1,Other,Other,True,Correct classification check,The predicted class agrees with the manually c...,Continue validation with more points and indep...
3,4,2,Built-up / Urban,Built-up / Urban,True,Correct classification check,The predicted class agrees with the manually c...,Continue validation with more points and indep...
4,5,3,Vegetation / Agriculture,Vegetation / Agriculture,True,Correct classification check,The predicted class agrees with the manually c...,Continue validation with more points and indep...
5,6,4,Other,Other,True,Correct classification check,The predicted class agrees with the manually c...,Continue validation with more points and indep...
6,7,5,Vegetation / Agriculture,Vegetation / Agriculture,True,Correct classification check,The predicted class agrees with the manually c...,Continue validation with more points and indep...
7,8,6,Vegetation / Agriculture,Vegetation / Agriculture,True,Correct classification check,The predicted class agrees with the manually c...,Continue validation with more points and indep...
8,9,7,Built-up / Urban,Built-up / Urban,True,Correct classification check,The predicted class agrees with the manually c...,Continue validation with more points and indep...
9,10,8,Vegetation / Agriculture,Vegetation / Agriculture,True,Correct classification check,The predicted class agrees with the manually c...,Continue validation with more points and indep...


In [94]:
from google.colab import files

files.download(
    "error_analysis_15_cases_completed.csv"
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [96]:
import ee
import pandas as pd
from IPython.display import display

# Function to calculate area of each class
def calculate_class_areas(classified_image, year, aoi):

    area_image = (
        ee.Image.pixelArea()
        .divide(1_000_000)
        .rename("area_km2")
        .addBands(
            classified_image
            .rename("landcover")
        )
    )

    grouped_result = area_image.reduceRegion(
        reducer=ee.Reducer.sum().group(
            groupField=1,
            groupName="class"
        ),
        geometry=aoi,
        scale=10,
        maxPixels=1e13,
        tileScale=4
    )

    groups = grouped_result.get("groups").getInfo()

    area_values = {
        0: 0.0,
        1: 0.0,
        2: 0.0
    }

    for item in groups:
        class_id = int(item["class"])
        area_values[class_id] = float(item["sum"])

    return {
        "Year": year,
        "Other (km2)": area_values[0],
        "Vegetation / Agriculture (km2)": area_values[1],
        "Built-up / Urban (km2)": area_values[2]
    }


# Calculate area statistics for both years
area_2019 = calculate_class_areas(
    classified_2019,
    2019,
    aoi
)

area_2025 = calculate_class_areas(
    classified_2025,
    2025,
    aoi
)

# Create detailed area table
area_table = pd.DataFrame([
    area_2019,
    area_2025
])

print("Land-cover area statistics:")
display(area_table.round(2))


Land-cover area statistics:


,Year,Other (km2),Vegetation / Agriculture (km2),Built-up / Urban (km2)
0,2019,500.74,4643.94,803.89
1,2025,475.11,4634.47,838.99


In [97]:
# Get total AOI area
total_aoi_area = float(aoi_area_km2)

# Get vegetation area from 2019
vegetation_2019_area = float(
    area_table.loc[
        area_table["Year"] == 2019,
        "Vegetation / Agriculture (km2)"
    ].iloc[0]
)

# Get built-up area from 2025
builtup_2025_area = float(
    area_table.loc[
        area_table["Year"] == 2025,
        "Built-up / Urban (km2)"
    ].iloc[0]
)

# Use the already calculated change area
conversion_area = float(
    estimated_change_km2 or 0
)

# Calculate affected percentage
affected_percentage = (
    conversion_area / total_aoi_area
) * 100

# Create final summary table
summary_table = pd.DataFrame({
    "Metric": [
        "Total AOI area",
        "Vegetation / Agriculture area in 2019",
        "Built-up / Urban area in 2025",
        "Estimated vegetation-to-built-up conversion",
        "Percentage of AOI affected"
    ],
    "Value": [
        round(total_aoi_area, 2),
        round(vegetation_2019_area, 2),
        round(builtup_2025_area, 2),
        round(conversion_area, 2),
        round(affected_percentage, 2)
    ],
    "Unit": [
        "km2",
        "km2",
        "km2",
        "km2",
        "%"
    ]
})

print("Punjab GeoGuardian Final Summary")
display(summary_table)


Punjab GeoGuardian Final Summary


,Metric,Value,Unit
0,Total AOI area,5953.79,km2
1,Vegetation / Agriculture area in 2019,4643.94,km2
2,Built-up / Urban area in 2025,838.99,km2
3,Estimated vegetation-to-built-up conversion,344.84,km2
4,Percentage of AOI affected,5.79,%


In [98]:
import geemap

# Create a final change-detection map
change_map = geemap.Map(
    basemap="HYBRID"
)

change_map.centerObject(
    aoi,
    8
)

# 2019 classified land cover
change_map.addLayer(
    classified_2019,
    {
        "min": 0,
        "max": 2,
        "palette": [
            "0000FF",
            "00AA00",
            "FF0000"
        ]
    },
    "Land Classification 2019",
    False
)

# 2025 classified land cover
change_map.addLayer(
    classified_2025,
    {
        "min": 0,
        "max": 2,
        "palette": [
            "0000FF",
            "00AA00",
            "FF0000"
        ]
    },
    "Land Classification 2025",
    False
)

# Vegetation-to-built-up conversion
change_map.addLayer(
    vegetation_to_built_up.selfMask(),
    {
        "palette": ["FF00FF"]
    },
    "Estimated Vegetation-to-Built-up Conversion",
    True
)

# AOI boundary
change_map.addLayer(
    faisalabad.style(
        color="FFFFFF",
        fillColor="00000000",
        width=2
    ),
    {},
    "Faisalabad AOI Boundary",
    True
)

change_map.add_layer_control()

display(change_map)


Map(center=[31.238674144990576, 73.14786051904862], controls=(WidgetControl(options=['position', 'transparent_…

In [102]:
import geemap
from IPython.display import display

# Create the map normally
dashboard_map = geemap.Map()

# Remove the default online basemap
dashboard_map.clear_layers()

# Center on the Faisalabad AOI
dashboard_map.centerObject(aoi, 8)

# RGB visualization
rgb_visualization = {
    "bands": ["B4", "B3", "B2"],
    "min": 0,
    "max": 3000
}

# NDVI visualization
ndvi_visualization = {
    "min": -0.5,
    "max": 0.9,
    "palette": [
        "8c510a",
        "d8b365",
        "f6e8c3",
        "c7eae5",
        "5ab4ac",
        "01665e"
    ]
}

# NDBI visualization
ndbi_visualization = {
    "min": -0.5,
    "max": 0.5,
    "palette": [
        "2166ac",
        "67a9cf",
        "d1e5f0",
        "fddbc7",
        "ef8a62",
        "b2182b"
    ]
}

# Classification visualization
classification_visualization = {
    "min": 0,
    "max": 2,
    "palette": [
        "0000FF",
        "00AA00",
        "FF0000"
    ]
}

# Add Earth Engine satellite imagery
dashboard_map.addLayer(
    composite_2025_clean,
    rgb_visualization,
    "Sentinel-2 RGB 2025",
    True
)

dashboard_map.addLayer(
    composite_2019_clean,
    rgb_visualization,
    "Sentinel-2 RGB 2019",
    False
)

# Add NDVI layers
dashboard_map.addLayer(
    ndvi_2019,
    ndvi_visualization,
    "NDVI 2019",
    False
)

dashboard_map.addLayer(
    ndvi_2025,
    ndvi_visualization,
    "NDVI 2025",
    False
)

# Add NDBI layers
dashboard_map.addLayer(
    ndbi_2019,
    ndbi_visualization,
    "NDBI 2019",
    False
)

dashboard_map.addLayer(
    ndbi_2025,
    ndbi_visualization,
    "NDBI 2025",
    False
)

# Add classification layers
dashboard_map.addLayer(
    classified_2019,
    classification_visualization,
    "Classification 2019",
    False
)

dashboard_map.addLayer(
    classified_2025,
    classification_visualization,
    "Classification 2025",
    False
)

# Add change detection layer
dashboard_map.addLayer(
    vegetation_to_built_up.selfMask(),
    {"palette": ["FF00FF"]},
    "Possible Vegetation-to-Built-up Change",
    True
)

# Add AOI boundary
dashboard_map.addLayer(
    faisalabad.style(
        color="FFFFFF",
        fillColor="00000000",
        width=2
    ),
    {},
    "Faisalabad AOI Boundary",
    True
)

# Add layer control
dashboard_map.add_layer_control()

# Display the map
display(dashboard_map)


Map(center=[31.238674144990576, 73.14786051904862], controls=(WidgetControl(options=['position', 'transparent_…

In [103]:
dashboard_map.to_html(
    "Punjab_GeoGuardian_Interactive_Dashboard.html"
)

print("Interactive dashboard saved successfully.")


Interactive dashboard saved successfully.


In [104]:
from google.colab import files

files.download(
    "Punjab_GeoGuardian_Interactive_Dashboard.html"
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [105]:
readme_text = """
# Punjab GeoGuardian

## AI-Based Monitoring of Urban Expansion and Farmland Loss

Punjab GeoGuardian is an educational GeoAI prototype that uses Sentinel-2 satellite imagery and machine learning to analyze possible land-use changes in Faisalabad District, Punjab, Pakistan.

The project compares satellite imagery from 2019 and 2025 and identifies possible changes from vegetation/agricultural land to built-up/urban land.

## Project Area

- Area of Interest: Faisalabad District
- Province: Punjab, Pakistan
- Total AOI area: 5953.79 km2
- Years compared: 2019 and 2025

## Main Objectives

- Analyze vegetation and agricultural land
- Detect built-up and urban areas
- Calculate NDVI and NDBI
- Apply Random Forest classification
- Detect possible vegetation-to-built-up conversion
- Calculate approximate area statistics
- Validate the classification using manually checked points
- Create an interactive map dashboard

## Data and Tools

- Google Colab
- Google Earth Engine
- Sentinel-2 Surface Reflectance Harmonized
- Python
- Geemap
- Folium
- Pandas
- NumPy
- Scikit-learn
- Random Forest

## Land-Cover Classes

| Class ID | Class |
|---:|---|
| 0 | Other |
| 1 | Vegetation / Agriculture |
| 2 | Built-up / Urban |

## Remote-Sensing Indices

### NDVI

NDVI was used to identify vegetation and agricultural areas.

NDVI = (NIR - Red) / (NIR + Red)

### NDBI

NDBI was used to support built-up and urban-area identification.

NDBI = (SWIR - NIR) / (SWIR + NIR)

## Results

| Metric | Value |
|---|---:|
| Total AOI area | 5953.79 km2 |
| Vegetation / Agriculture area in 2019 | 4643.94 km2 |
| Built-up / Urban area in 2025 | 838.99 km2 |
| Estimated vegetation-to-built-up conversion | 344.84 km2 |
| Percentage of AOI affected | 5.79% |
| Validation points | 60 |
| Validation accuracy | 96.67% |

## Important Interpretation

The estimated vegetation-to-built-up conversion represents pixels classified as vegetation or agriculture in 2019 and built-up or urban in 2025.

These results are estimated satellite-based outputs. They should not be treated as official land records, legal evidence, or an official government monitoring system.

## Project Outputs

- Punjab GeoGuardian Google Colab notebook
- RGB maps for 2019 and 2025
- NDVI maps for 2019 and 2025
- NDBI maps for 2019 and 2025
- Land classification maps
- Vegetation-to-built-up change map
- Area statistics tables
- Validation results
- Error-analysis file
- Interactive HTML dashboard
- Final project report

## Limitations

- Sentinel-2 imagery has a spatial resolution of approximately 10 metres for key bands.
- Mixed pixels may cause classification errors.
- Dry soil can be confused with built-up land.
- Seasonal crop changes may affect NDVI.
- Small buildings and narrow roads may not be detected.
- Image dates and atmospheric conditions may differ.
- More manually verified training and validation samples would improve reliability.

## Future Improvements

- Add more land-cover classes
- Use higher-resolution imagery
- Compare multiple years
- Add district-level monitoring
- Use official reference datasets
- Test segmentation models such as U-Net
- Build a Streamlit dashboard
- Add automated PDF reporting

## Disclaimer

This project was developed for educational and internship purposes. It is a learning prototype and is not an official government monitoring or decision-support system.
"""

with open(
    "Punjab_GeoGuardian_README.md",
    "w",
    encoding="utf-8"
) as file:
    file.write(readme_text)

print("README created successfully.")


README created successfully.


In [106]:
from google.colab import files

files.download(
    "Punjab_GeoGuardian_README.md"
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [107]:
requirements_text = """earthengine-api
geemap
folium
rasterio
geopandas
scikit-learn
pandas
numpy
matplotlib
seaborn
"""

with open(
    "requirements.txt",
    "w",
    encoding="utf-8"
) as file:
    file.write(requirements_text)

print("requirements.txt created successfully.")


requirements.txt created successfully.


In [108]:
from google.colab import files

files.download(
    "requirements.txt"
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [110]:
linkedin_summary = """
I completed AIRI Team PITB AI Internship Task 3 by developing Punjab GeoGuardian, an educational GeoAI prototype for monitoring possible urban expansion and farmland loss in Faisalabad District, Punjab, Pakistan.

The project compares Sentinel-2 satellite imagery from 2019 and 2025. It uses Google Earth Engine, Python, NDVI, NDBI, and a Random Forest classifier to classify land into:

- Other
- Vegetation / Agriculture
- Built-up / Urban

The workflow includes:

- Area of Interest selection
- Sentinel-2 satellite data collection
- Cloud masking and image compositing
- NDVI and NDBI calculation
- Random Forest land-cover classification
- Change detection
- Estimated vegetation-to-built-up conversion
- Area statistics calculation
- 60-point validation
- Error analysis
- Interactive map dashboard

Key estimated results:

- Total AOI area: 5,953.79 km2
- Estimated vegetation-to-built-up conversion: 344.84 km2
- Estimated percentage of AOI affected: 5.79%
- Validation points checked: 60
- Validation accuracy: 96.67%

Tools used:

Python, Google Colab, Google Earth Engine, Sentinel-2, Geemap, Folium, Pandas, NumPy, Scikit-learn, and Random Forest.

This project helped me understand how AI and satellite imagery can support urban planning, agricultural monitoring, environmental analysis, and public-sector decision support.

Important note: This is an educational prototype. The detected changes are estimated satellite-based results and are not official land records or government monitoring outputs.

#GeoAI #RemoteSensing #SatelliteImagery #MachineLearning #UrbanPlanning #Agriculture #Python #GoogleEarthEngine #AIRI #PITB #AIInternship
"""

with open(
    "Punjab_GeoGuardian_LinkedIn_Summary.txt",
    "w",
    encoding="utf-8"
) as file:
    file.write(linkedin_summary)

print("LinkedIn summary created successfully.")


LinkedIn summary created successfully.


In [111]:
from google.colab import files

files.download(
    "Punjab_GeoGuardian_LinkedIn_Summary.txt"
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [112]:
import os
import zipfile
from pathlib import Path

# Create a folder for final submission files
submission_folder = Path("Punjab_GeoGuardian_Final_Submission")
submission_folder.mkdir(exist_ok=True)

# Files that should be included
files_to_include = [
    "Punjab_GeoGuardian.ipynb",
    "Punjab_GeoGuardian_completed.ipynb",
    "Punjab_GeoGuardian_Final_Report.md",
    "Punjab_GeoGuardian_README.md",
    "Punjab_GeoGuardian_Interactive_Dashboard.html",
    "Punjab_GeoGuardian_LinkedIn_Summary.txt",
    "requirements.txt",
    "validation_points_completed.csv",
    "validation_results.csv",
    "error_analysis_15_cases_completed.csv",
    "land_cover_area_statistics.csv",
    "project_summary_statistics.csv"
]

# Copy available files into the submission folder
copied_files = []

for file_name in files_to_include:
    if os.path.exists(file_name):
        destination = submission_folder / file_name

        with open(file_name, "rb") as source_file:
            with open(destination, "wb") as destination_file:
                destination_file.write(source_file.read())

        copied_files.append(file_name)

print("Files copied to submission folder:")
for file_name in copied_files:
    print("-", file_name)

# Create ZIP file
zip_name = "Punjab_GeoGuardian_Final_Submission.zip"

with zipfile.ZipFile(
    zip_name,
    "w",
    zipfile.ZIP_DEFLATED
) as zip_file:

    for file_path in submission_folder.iterdir():
        zip_file.write(
            file_path,
            arcname=file_path.name
        )

print("\nZIP file created successfully:")
print(zip_name)

print("\nFiles not found and skipped:")
for file_name in files_to_include:
    if file_name not in copied_files:
        print("-", file_name)


Files copied to submission folder:
- Punjab_GeoGuardian_README.md
- Punjab_GeoGuardian_Interactive_Dashboard.html
- Punjab_GeoGuardian_LinkedIn_Summary.txt
- requirements.txt
- validation_points_completed.csv
- error_analysis_15_cases_completed.csv

ZIP file created successfully:
Punjab_GeoGuardian_Final_Submission.zip

Files not found and skipped:
- Punjab_GeoGuardian.ipynb
- Punjab_GeoGuardian_completed.ipynb
- Punjab_GeoGuardian_Final_Report.md
- validation_results.csv
- land_cover_area_statistics.csv
- project_summary_statistics.csv


In [113]:
from google.colab import files

files.download(
    "Punjab_GeoGuardian_Final_Submission.zip"
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [114]:
import zipfile
import os

zip_name = "Punjab_GeoGuardian_Final_Submission.zip"

if not os.path.exists(zip_name):
    print("ZIP file not found. Please create it first.")
else:
    with zipfile.ZipFile(zip_name, "r") as zip_file:
        files_inside = zip_file.namelist()

    print("ZIP verification successful.")
    print("Files inside ZIP:")

    for file_name in files_inside:
        print("-", file_name)

    print("\nTotal files:", len(files_inside))



ZIP verification successful.
Files inside ZIP:
- Punjab_GeoGuardian_README.md
- Punjab_GeoGuardian_LinkedIn_Summary.txt
- error_analysis_15_cases_completed.csv
- validation_points_completed.csv
- requirements.txt
- Punjab_GeoGuardian_Interactive_Dashboard.html

Total files: 6
